# 06g: PM100 Sequence-Tie Variant (skip_back)

PM100-only exploration of an alternative sequence-construction policy for LSTM/TCN history. The
adopted construction (`src/sequences.build_sequence_index`, default `tie_policy="exclude"`,
documented in `06a_sequence_investigation.ipynb`) excludes a same-user candidate history
position outright when it ties the target job's own submit timestamp -- which `06a`'s real
mask-based diagnostic found cuts the usable history roughly in half on both datasets. This
notebook adds an optional `tie_policy="skip_back"` to the same function (default unchanged,
verified byte-identical below) that instead looks PAST a tied candidate for the next
strictly-earlier same-user job, and trains LSTM/TCN under it for all three PM100 combinations to
see whether that recovers any of the LSTM/TCN-vs-classical-ML gap.

Scope, stated plainly:
- PM100 only. F-DATA is NOT retrained here -- only its padding share under `skip_back` is
  measured below (index + mask only, no model training), to show whether a full F-DATA rerun
  would even be worth the cost.
- FNN does not use sequence history at all, so it is not rerun here; the original FNN result
  (from `06c_dl_power.ipynb`/`06d_dl_duration.ipynb`/`06e_dl_memory.ipynb`) is used as-is for
  comparison.
- Every original checkpoint, Optuna study, and results file is untouched: this notebook writes
  to its own checkpoint directory, its own Optuna studies (`_skipback` suffix), and its own
  result files (`_skipback` suffix) -- verified below to never collide with the originals.

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be set before any CUDA/torch init

import sys
sys.path.append("..")

import glob
import pathlib
import random
import shutil
import time
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import optuna
import pandas as pd
import torch
from IPython.display import display

from src import features, sequences
from src.dl_pipeline import (prepare_combo_data, run_pair, load_pair_artifacts,
                              save_pair_artifacts, STUDIES_DIR)

pd.set_option("display.width", 160)

SEED = 0
CKPT_DIR_SKIPBACK = pathlib.Path("../data/interim/notebook06g_skipback_models")
RESULTS_DIR_SKIPBACK = pathlib.Path("../data/interim/notebook06g_skipback_results")
CKPT_DIR_SKIPBACK.mkdir(parents=True, exist_ok=True)


def set_determinism(seed=SEED):
    """Called before every training run in this notebook, not just once at the top -- re-seeding
    immediately before each run is what the determinism check below actually tests."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


set_determinism()
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"device={DEVICE}, CUBLAS_WORKSPACE_CONFIG={os.environ.get('CUBLAS_WORKSPACE_CONFIG')}")

device=cuda:0, CUBLAS_WORKSPACE_CONFIG=:4096:8


## Default behavior unchanged: equivalence check against the original implementation

`src/sequences.build_sequence_index` gained an optional `tie_policy` parameter ("exclude",
the default, or "skip_back") for this notebook. The cell below keeps a verbatim copy of the
function exactly as it existed before that change -- the same documentation convention
`06b_model_architectures.ipynb` uses for its own verbatim class copies -- and asserts the new
function's DEFAULT output is byte-identical to it on both real datasets.

In [2]:
def build_sequence_index_ORIGINAL(user_ids, times):
    """Verbatim copy of src.sequences.build_sequence_index as it existed before the
    tie_policy parameter was added -- kept here only to prove the new function's default
    behavior is unchanged."""
    SEQ_LEN = sequences.SEQ_LEN
    n = len(user_ids)
    df = pd.DataFrame({"user_id": np.asarray(user_ids), "time": pd.to_datetime(times),
                        "orig_row": np.arange(n)})
    df = df.sort_values(["user_id", "time"], kind="mergesort").reset_index(drop=True)

    seq_idx = np.full((n, SEQ_LEN), -1, dtype=np.int64)
    mask = np.zeros((n, SEQ_LEN), dtype=bool)

    orig_row = df["orig_row"].to_numpy()
    times_sorted = df["time"].to_numpy()
    pos_in_user = df.groupby("user_id", sort=False).cumcount().to_numpy()

    for k in range(1, SEQ_LEN + 1):
        src_sorted_pos = np.arange(n) - k
        valid = pos_in_user >= k
        valid &= np.where(pos_in_user >= k, times_sorted[np.clip(src_sorted_pos, 0, None)],
                           times_sorted[0]) < times_sorted
        dest_col = SEQ_LEN - k
        rows_valid = np.where(valid)[0]
        seq_idx[orig_row[rows_valid], dest_col] = orig_row[src_sorted_pos[rows_valid]]
        mask[orig_row[rows_valid], dest_col] = True
    return seq_idx, mask

In [3]:
PM100_PATH = "../data/raw/pm100/pm100_job_table.parquet"
pm100_base = pd.read_parquet(PM100_PATH)
pm100_base = features.filter_completed_jobs(pm100_base, "pm100")
print(f"PM100: {len(pm100_base):,} completed jobs")

t0 = time.perf_counter()
FDATA_DIR = "../data/raw/fdata"
fdata_files = sorted(glob.glob(f"{FDATA_DIR}/*.parquet"))[:38]
fdata_base = features.load_fdata_no_embedding(fdata_files)
fdata_base = features.filter_completed_jobs(fdata_base, "fdata")
print(f"F-DATA loaded in {time.perf_counter() - t0:.0f}s: {len(fdata_base):,} completed jobs "
      f"(index + mask only below -- no F-DATA training in this notebook)")

[pm100] excluded 22.0% of jobs as non-completed (50928 / 231238)
PM100: 180,310 completed jobs


[fdata] excluded 9.9% of jobs as non-completed (2558519 / 25866900)
F-DATA loaded in 18s: 23,308,381 completed jobs (index + mask only below -- no F-DATA training in this notebook)


In [4]:
new_pm100_idx, new_pm100_mask = sequences.build_sequence_index(
    pm100_base["user_id"].to_numpy(), pm100_base["submit_time"])  # default tie_policy="exclude"
orig_pm100_idx, orig_pm100_mask = build_sequence_index_ORIGINAL(
    pm100_base["user_id"].to_numpy(), pm100_base["submit_time"])
pm100_equal = (np.array_equal(new_pm100_idx, orig_pm100_idx)
               and np.array_equal(new_pm100_mask, orig_pm100_mask))

new_fdata_idx, new_fdata_mask = sequences.build_sequence_index(
    fdata_base["usr"].to_numpy(), fdata_base["adt"])
orig_fdata_idx, orig_fdata_mask = build_sequence_index_ORIGINAL(
    fdata_base["usr"].to_numpy(), fdata_base["adt"])
fdata_equal = (np.array_equal(new_fdata_idx, orig_fdata_idx)
               and np.array_equal(new_fdata_mask, orig_fdata_mask))

print(f"PM100 default (tie_policy='exclude') == original implementation: {pm100_equal}")
print(f"F-DATA default (tie_policy='exclude') == original implementation: {fdata_equal}")
assert pm100_equal and fdata_equal, "default tie_policy must be byte-identical to the original"
print("EQUIVALENCE CONFIRMED on both real datasets -- the default is unchanged")

del new_pm100_idx, orig_pm100_idx, new_fdata_idx, orig_fdata_idx

PM100 default (tie_policy='exclude') == original implementation: True
F-DATA default (tie_policy='exclude') == original implementation: True
EQUIVALENCE CONFIRMED on both real datasets -- the default is unchanged


## Real padding share under `skip_back` (same mask-based method as `06a_sequence_investigation.ipynb`)

For PM100 this is a real, mask-based measurement on the actual real-sequence index (not the
count-based proxy `06a` warns against). For F-DATA, this is index + mask construction only -- no
model training -- specifically to show whether a full F-DATA rerun would be worth the cost.

In [5]:
def diagnose_padding(name, df, user_col, mask, highlight_users=None):
    real_per_job = mask.sum(axis=1)
    print(f"\n=== {name} (skip_back) ===")
    display(pd.Series(real_per_job).value_counts().sort_index().to_frame("n_jobs"))
    print(f"fully unpadded (== {sequences.SEQ_LEN}): {(real_per_job == sequences.SEQ_LEN).mean():.2%}")
    print(f"all padding (== 0): {(real_per_job == 0).mean():.2%}")
    print(f"mean real positions: {real_per_job.mean():.4f}")
    if highlight_users:
        user_arr = df[user_col].to_numpy()
        rows = []
        for u in highlight_users:
            m = user_arr == u
            if m.sum() == 0:
                continue
            rows.append(dict(user=u, n_jobs=int(m.sum()),
                              fully_unpadded_pct=(real_per_job[m] == sequences.SEQ_LEN).mean() * 100,
                              mean_real_positions=real_per_job[m].mean()))
        display(pd.DataFrame(rows).set_index("user").round(2))
    return real_per_job


pm100_skip_idx, pm100_skip_mask = sequences.build_sequence_index(
    pm100_base["user_id"].to_numpy(), pm100_base["submit_time"], tie_policy="skip_back")
pm100_skip_real = diagnose_padding("PM100", pm100_base, "user_id", pm100_skip_mask)

n_rows, n_pos = sequences.verify_no_leakage(
    pm100_base["user_id"].to_numpy(), pm100_base["submit_time"], pm100_skip_idx, pm100_skip_mask)
print(f"\nPM100 skip_back leakage test (same check 06a uses): {n_rows:,} rows, "
      f"{n_pos:,} positions checked, PASSED")


=== PM100 (skip_back) ===


,n_jobs
0,661
1,1107
2,386
3,363
4,349
5,342
6,341
7,300
8,298
9,327


fully unpadded (== 10): 97.52%
all padding (== 0): 0.37%
mean real positions: 9.8381



PM100 skip_back leakage test (same check 06a uses): 180,310 rows, 1,773,907 positions checked, PASSED


In [6]:
t0 = time.perf_counter()
fdata_skip_idx, fdata_skip_mask = sequences.build_sequence_index(
    fdata_base["usr"].to_numpy(), fdata_base["adt"], tie_policy="skip_back")
print(f"F-DATA skip_back index built in {time.perf_counter() - t0:.0f}s (index + mask only, no training)")

fdata_skip_real = diagnose_padding("F-DATA", fdata_base, "usr", fdata_skip_mask,
                                    highlight_users=["usr_1794", "usr_1848", "usr_2853"])

n_rows2, n_pos2 = sequences.verify_no_leakage(
    fdata_base["usr"].to_numpy(), fdata_base["adt"], fdata_skip_idx, fdata_skip_mask,
    sample_size=200_000, seed=0)
print(f"\nF-DATA skip_back leakage test (200k-row sample): {n_rows2:,} rows, "
      f"{n_pos2:,} positions checked, PASSED")

del fdata_base, fdata_skip_idx, fdata_skip_mask, new_fdata_mask, orig_fdata_mask
print("\nF-DATA data released from memory (PM100-only training follows -- no F-DATA rerun here)")

F-DATA skip_back index built in 128s (index + mask only, no training)



=== F-DATA (skip_back) ===


,n_jobs
0,4241
1,15569
2,4571
3,2888
4,3507
5,2679
6,2712
7,2526
8,2501
9,2844


fully unpadded (== 10): 99.81%
all padding (== 0): 0.02%
mean real positions: 9.9871


,n_jobs,fully_unpadded_pct,mean_real_positions
user,,,
usr_1794,176802,99.99,10.0
usr_1848,150712,99.99,10.0
usr_2853,42138,99.98,10.0



F-DATA skip_back leakage test (200k-row sample): 200,000 rows, 1,997,430 positions checked, PASSED

F-DATA data released from memory (PM100-only training follows -- no F-DATA rerun here)


**Padding-share findings, read against `06a_sequence_investigation.ipynb`'s `exclude`-policy numbers (~51% fully unpadded, ~17-19% all-padding, both datasets):**

`skip_back` essentially eliminates the padding problem `06a` diagnosed. On PM100, fully-unpadded jumps from 51.5% (`exclude`) to **97.52%**, all-padding drops from 19.2% to **0.37%**, and the mean real positions per job rises from roughly half-full to **9.84 of 10**. On F-DATA the effect is even larger: fully-unpadded rises from 50.7% to **99.81%**, all-padding drops from 17.1% to **0.02%**, mean real positions **9.99 of 10**.

The three F-DATA users singled out in `06a` and `06f_dl_comparison.ipynb` (`usr_1794`, `usr_1848`, `usr_2853` -- whose thin, heavily-padded history under `exclude` tracked almost exactly with how badly their rows drove LSTM/TCN's raw-space collapse) go from 17.44%/45.13%/67.08% fully unpadded (mean real positions 2.30/5.80/7.49) under `exclude` to **99.99%/99.99%/99.98% fully unpadded, mean real positions 10.0/10.0/10.0** under `skip_back`. The padding that was concentrated on exactly these users is gone.

This is a real, mask-based measurement for PM100 (leakage-test PASSED, 1,773,907 positions checked) and an index+mask-only measurement for F-DATA (no training; leakage-test PASSED on a 200k-row sample, 1,997,430 positions checked) -- confirming a full F-DATA rerun would very plausibly be worth doing, though it is out of scope here (PM100 only, per the scope above).

## Determinism check: one TCN pair, run twice with the same seed

PM100 power's TCN pair (the cheapest TCN pair to train) is trained TWICE from scratch, under a
fresh Optuna study each time (same seed, same seeded `TPESampler`), with the determinism
settings above already active. If the settings work, the two runs should be bit-identical; if
not, this says so plainly and reports which operation(s) warned.

In [7]:
DET_CHECK_COMBO = "pm100_power"
prepared_det = prepare_combo_data(DET_CHECK_COMBO, tie_policy="skip_back")
det_ckpt_dir = CKPT_DIR_SKIPBACK / "determinism_check"
det_study_db = STUDIES_DIR / f"{DET_CHECK_COMBO}_TCN_detcheck.db"


def run_tcn_once(seed):
    set_determinism(seed)
    if det_study_db.exists():
        det_study_db.unlink()
    if det_ckpt_dir.exists():
        shutil.rmtree(det_ckpt_dir)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = run_pair(prepared_det, "TCN", device=DEVICE, ckpt_dir=det_ckpt_dir,
                           study_suffix="_detcheck", sampler=optuna.samplers.TPESampler(seed=seed))
    det_warnings = [str(wi.message) for wi in caught
                    if "determinist" in str(wi.message).lower()]
    return result, det_warnings


t0 = time.perf_counter()
result_a, warnings_a = run_tcn_once(SEED)
print(f"run A done in {time.perf_counter() - t0:.1f}s")

t0 = time.perf_counter()
result_b, warnings_b = run_tcn_once(SEED)
print(f"run B done in {time.perf_counter() - t0:.1f}s")

[pm100] excluded 22.0% of jobs as non-completed (50928 / 231238)


pm100_power: 126,217 train / 54,093 test rows; scaler log-transformed 7/28 columns; sequences built in 2s, fill rate 98.38%


[I 2026-11-02 12:15:07,891] A new study created in RDB with name: pm100_power_TCN_detcheck


[I 2026-11-02 12:16:08,706] Trial 0 finished with value: 0.19217402273483575 and parameters: {'lr': 0.00012520653814999459}. Best is trial 0 with value: 0.19217402273483575.


[I 2026-11-02 12:17:10,481] Trial 1 finished with value: 0.08908458760991336 and parameters: {'lr': 0.0002693883019285409}. Best is trial 1 with value: 0.08908458760991336.


[I 2026-11-02 12:18:12,186] Trial 2 finished with value: 0.13736110574572763 and parameters: {'lr': 0.00016051911333587616}. Best is trial 1 with value: 0.08908458760991336.


... (16 outputs left out)

[I 2026-11-02 12:33:34,940] Trial 7 finished with value: 0.08044581647447013 and parameters: {'lr': 0.0006074996073425694}. Best is trial 7 with value: 0.08044581647447013.


[I 2026-11-02 12:34:27,612] Trial 8 finished with value: 0.08405563717427901 and parameters: {'lr': 0.0008459126528049375}. Best is trial 7 with value: 0.08044581647447013.


[I 2026-11-02 12:35:29,904] Trial 9 finished with value: 0.19142330763145815 and parameters: {'lr': 5.8463261216434157e-05}. Best is trial 7 with value: 0.08044581647447013.


run B done in 613.2s


In [8]:
preds_equal = np.array_equal(result_a["preds_log"], result_b["preds_log"])
r2_equal = (result_a["raw_metrics"]["R2"] == result_b["raw_metrics"]["R2"]
            and result_a["log_metrics"]["R2"] == result_b["log_metrics"]["R2"])

print(f"Run A: log_R2={result_a['log_metrics']['R2']:.10f} raw_R2={result_a['raw_metrics']['R2']:.10f} "
      f"best_lr={result_a['best_lr']:.6e}")
print(f"Run B: log_R2={result_b['log_metrics']['R2']:.10f} raw_R2={result_b['raw_metrics']['R2']:.10f} "
      f"best_lr={result_b['best_lr']:.6e}")
print(f"\nTest predictions bit-identical (np.array_equal): {preds_equal}")
print(f"R2 (log and raw) identical to full float precision: {r2_equal}")

all_det_warnings = sorted(set(warnings_a + warnings_b))
if all_det_warnings:
    print(f"\n{len(all_det_warnings)} distinct determinism warning(s) from "
          f"torch.use_deterministic_algorithms(warn_only=True):")
    for msg in all_det_warnings:
        print(f"  - {msg}")
else:
    print("\nNo determinism warnings were raised.")

DETERMINISTIC = bool(preds_equal and r2_equal)
print(f"\nCONCLUSION: the two runs are {'BIT-IDENTICAL' if DETERMINISTIC else 'NOT bit-identical'}.")

Run A: log_R2=0.9269904493 raw_R2=0.8716793807 best_lr=6.074996e-04
Run B: log_R2=0.9269904493 raw_R2=0.8716793807 best_lr=6.074996e-04

Test predictions bit-identical (np.array_equal): True
R2 (log and raw) identical to full float precision: True

No determinism warnings were raised.

CONCLUSION: the two runs are BIT-IDENTICAL.


## Training: PM100 LSTM/TCN under `skip_back`, new Optuna studies

For each of PM100 power, duration, and memory: a fresh 10-trial Optuna study (learning rate
1e-5 to 1e-3 log-uniform, seeded `TPESampler`), the same split/features/scaling/batch
size/epochs/patience as the original sweep (all reused unchanged from `src/dl_pipeline`), the
only difference being `tie_policy="skip_back"` in sequence construction and a fresh checkpoint
directory + Optuna study (`_skipback` suffix). FNN is not retrained (it ignores sequence
history entirely); the original FNN result is read back below for comparison.

In [9]:
PM100_COMBOS = ["pm100_power", "pm100_duration", "pm100_memory_c4"]
SKIPBACK_ARCHS = ["LSTM", "TCN"]

skipback_results = {}
skipback_prepared = {}
for combo in PM100_COMBOS:
    t0 = time.perf_counter()
    prepared = prepare_combo_data(combo, tie_policy="skip_back")
    skipback_prepared[combo] = prepared
    print(f"{combo}: prepared in {time.perf_counter() - t0:.0f}s, "
          f"fill rate {prepared['mask'].mean():.2%}")
    skipback_results[combo] = {}
    for arch in SKIPBACK_ARCHS:
        set_determinism(SEED)
        t1 = time.perf_counter()
        result = run_pair(prepared, arch, device=DEVICE, ckpt_dir=CKPT_DIR_SKIPBACK,
                           study_suffix="_skipback", sampler=optuna.samplers.TPESampler(seed=SEED))
        print(f"  {arch}_skipback: best_trial=#{result['best_trial']} "
              f"best_lr={result['best_lr']:.2e} log_R2={result['log_metrics']['R2']:.4f} "
              f"raw_R2={result['raw_metrics']['R2']:.4f} ({time.perf_counter() - t1:.1f}s)")
        skipback_results[combo][arch] = result
        save_pair_artifacts(combo, f"{arch}_skipback", prepared, result, out_dir=RESULTS_DIR_SKIPBACK)

[I 2026-11-02 12:35:33,583] Using an existing study with name 'pm100_power_LSTM_skipback' instead of creating a new one.


pm100_power: 126,217 train / 54,093 test rows; scaler log-transformed 7/28 columns; sequences built in 2s, fill rate 98.38%
pm100_power: prepared in 2s, fill rate 98.38%


[I 2026-11-02 12:35:34,858] Using an existing study with name 'pm100_power_TCN_skipback' instead of creating a new one.


  LSTM_skipback: best_trial=#8 best_lr=8.46e-04 log_R2=0.9381 raw_R2=0.9113 (1.3s)


  TCN_skipback: best_trial=#7 best_lr=6.07e-04 log_R2=0.9270 raw_R2=0.8717 (1.4s)


[I 2026-11-02 12:35:38,437] Using an existing study with name 'pm100_duration_LSTM_skipback' instead of creating a new one.


... (5 outputs left out)

pm100_memory_c4: 126,217 train / 54,093 test rows; scaler log-transformed 4/22 columns; sequences built in 2s, fill rate 98.38%
pm100_memory_c4: prepared in 2s, fill rate 98.38%


[I 2026-11-02 12:35:44,668] Using an existing study with name 'pm100_memory_c4_TCN_skipback' instead of creating a new one.


  LSTM_skipback: best_trial=#8 best_lr=8.46e-04 log_R2=0.9646 raw_R2=0.8932 (1.2s)


  TCN_skipback: best_trial=#8 best_lr=8.46e-04 log_R2=0.9667 raw_R2=0.8937 (1.4s)


## Results: original (`exclude`) vs `skip_back`, original FNN, and the same-features LightGBM control

Loaded from real artifacts, not hand-typed: original LSTM/TCN/FNN from
`data/interim/notebook06_results/` (written by `06c_dl_power.ipynb`/`06d_dl_duration.ipynb`/
`06e_dl_memory.ipynb`), the same-features control from
`results/notebook06_same_features_control_{combo}_LightGBM.csv` (written by
`06f_dl_comparison.ipynb`), and the `skip_back` results trained live above.

In [10]:
def load_control(combo, model="LightGBM"):
    path = pathlib.Path(f"../results/notebook06_same_features_control_{combo}_{model}.csv")
    control_df = pd.read_csv(path)
    out = {"log_metrics": {}, "raw_metrics": {}}
    for _, row in control_df.iterrows():
        key = "log_metrics" if row["space"] == "log" else "raw_metrics"
        out[key][row["metric"]] = row["value"]
    return out


def metric_row(combo, source, r):
    return dict(combo=combo, source=source, log_R2=r["log_metrics"]["R2"], raw_R2=r["raw_metrics"]["R2"],
                raw_MedAE=r["raw_metrics"]["MedAE"], raw_Within20pct=r["raw_metrics"]["Within20pct"])


comparison_rows = []
for combo in PM100_COMBOS:
    original_fnn = load_pair_artifacts(combo, "FNN")
    control = load_control(combo, "LightGBM")
    comparison_rows.append(metric_row(combo, "original_FNN", original_fnn))
    comparison_rows.append(metric_row(combo, "same_features_control_LightGBM", control))
    for arch in SKIPBACK_ARCHS:
        original = load_pair_artifacts(combo, arch)
        comparison_rows.append(metric_row(combo, f"original_{arch}", original))
        comparison_rows.append(metric_row(combo, f"skipback_{arch}", skipback_results[combo][arch]))

comparison_df = pd.DataFrame(comparison_rows).set_index(["combo", "source"])
display(comparison_df.round(4))

log_R2  raw_R2  raw_MedAE  raw_Within20pct
combo           source                                                                    
pm100_power     original_FNN                    0.9293  0.8945   149.5496          59.5290
                same_features_control_LightGBM  0.9451  0.9013   127.0169          67.5744
                original_LSTM                   0.9321  0.8845   138.2768          63.5960
                skipback_LSTM                   0.9381  0.9113   132.6991          63.8715
                original_TCN                    0.8873  0.8091   146.9198          53.8517
                skipback_TCN                    0.9270  0.8717   147.3774          60.3442
pm100_duration  original_FNN                    0.8162  0.6369    12.6341           9.0588
                same_features_control_LightGBM  0.8684  0.6858     9.9748          12.0747
                original_LSTM                   0.8775  0.4881     6.6894           9.8225
                skipback_LSTM                   0.9110  0.5033     3.6655          17.5333
                original_TCN                    0.8717  0.4768     6.9831           8.6409
                skipback_TCN                    0.8049  0.4740    13.5089          11.4608
pm100_memory_c4 original_FNN                    0.9764  0.9660     1.4105          95.2619
                same_features_control_LightGBM  0.9801  0.9676     1.6774          96.4894
                original_LSTM                   0.9621  0.9367     6.7138          89.0392
                skipback_LSTM                   0.9646  0.8932     3.0782          92.4944
                original_TCN                    0.9590  0.9010    21.2135          86.4067
                skipback_TCN                    0.9667  0.8937     7.3069          95.4578

In [11]:
SKIPBACK_RESULTS_CSV_DIR = pathlib.Path("../results")
COMBO_DATASET_TARGET = {"pm100_power": ("pm100", "power"), "pm100_duration": ("pm100", "duration"),
                         "pm100_memory_c4": ("pm100", "memory")}

written = []
for combo in PM100_COMBOS:
    dataset, target = COMBO_DATASET_TARGET[combo]
    for arch in SKIPBACK_ARCHS:
        r = skipback_results[combo][arch]
        rows = []
        for space, m in [("log", r["log_metrics"]), ("raw", r["raw_metrics"])]:
            for metric, value in m.items():
                rows.append(dict(
                    dataset=dataset, target=target, model=f"{arch}_skipback",
                    variant="sequence_tie_skip_back", tree_method="", fit_rows=r["fit_rows"],
                    seed=SEED, space=space, metric=metric, value=value, status="diagnostic",
                    source=(f"notebook06g_sequence_tie_variant: tie_policy='skip_back' "
                            f"(src/sequences.build_sequence_index), fresh Optuna study "
                            f"'{combo}_{arch}_skipback' (10 trials, lr 1e-5..1e-3 log-uniform, "
                            f"TPESampler seed={SEED}), best_trial=#{r['best_trial']}, "
                            f"best_lr={r['best_lr']:.4e}, fit_rows is tune_train; exploratory "
                            f"only, NOT read by scripts/build_results_ledger.py -- status "
                            f"'diagnostic', never 'adopted'"),
                    is_seed_sweep=False))
        out_path = SKIPBACK_RESULTS_CSV_DIR / f"notebook06g_sweep_result_{combo}_{arch}_skipback.csv"
        pd.DataFrame(rows).to_csv(out_path, index=False)
        written.append(str(out_path))

print(f"wrote {len(written)} result files:")
for w in written:
    print(f"  {w}")
print("\nNote: these are NOT read by scripts/build_results_ledger.py -- this variant is "
      "exploratory and does not change results/results_ledger.csv.")

wrote 6 result files:
  ../results/notebook06g_sweep_result_pm100_power_LSTM_skipback.csv
  ../results/notebook06g_sweep_result_pm100_power_TCN_skipback.csv
  ../results/notebook06g_sweep_result_pm100_duration_LSTM_skipback.csv
  ../results/notebook06g_sweep_result_pm100_duration_TCN_skipback.csv
  ../results/notebook06g_sweep_result_pm100_memory_c4_LSTM_skipback.csv
  ../results/notebook06g_sweep_result_pm100_memory_c4_TCN_skipback.csv

Note: these are NOT read by scripts/build_results_ledger.py -- this variant is exploratory and does not change results/results_ledger.csv.


## Summary: does `skip_back` change the DL-vs-classical conclusion?

**No -- not for the headline raw-R2 conclusion, on any of the three PM100 combinations, when compared to classical ML's own overall adopted best model** (XGBoost, raw R2 0.9243 for power; LightGBM, raw R2 0.6994 for duration; XGBoost, raw R2 0.9658 for memory -- `results/results_ledger.csv`, unaffected by anything in this notebook). The multi-seed check below settles which of the single-seed effects reported earlier are real and which were seed noise; this section is rewritten to match those real numbers, not the single-seed table alone.

**Determinism, now checked on two pairs**: PM100 power TCN (above) and PM100 duration TCN (in the multi-seed section below) were each trained twice from scratch with identical settings and matched to full float precision on every trial, the winning trial, the learning rate, and the test predictions, with zero determinism warnings either time. Both BIT-IDENTICAL.

**PM100 power, raw R2 -- real, not seed noise.** The 3-seed `skip_back` range for LSTM (0.9165-0.9214) sits entirely ABOVE the original (`exclude`) base-width range (0.8565-0.9131, 3 seeds, from `06f`) -- no overlap at all. The single seed-0 number reported earlier (0.9113) undersold this: with 3 seeds, `skip_back` LSTM's raw R2 is reliably above the same-features control (0.9013) and reliably above every seed of the original policy, though still below the overall adopted best (XGBoost, 0.9243). TCN's mean also moves up clearly (0.7899 -> 0.8262), though its range (0.7961-0.8539) overlaps the original's top end (0.7660-0.8059) slightly -- a likely real effect, just not as cleanly separated as LSTM's. Log R2 moved up clearly, non-overlapping, for both architectures (LSTM 0.9258-range -> 0.9379-range; TCN 0.8844-range -> 0.9126-range).

**PM100 duration -- the apparent raw-R2 move does NOT survive the multi-seed check.** `skip_back` LSTM's 3-seed raw R2 range (0.4867-0.6285) heavily overlaps the original's (0.4391-0.5086), and the std roughly doubled (0.0718 vs 0.0374) -- the single seed-0 result (0.5033) was unremarkable noise, not a real shift. TCN duration's raw R2 is the same story (0.4442-0.5526 vs 0.3886-0.4848, overlapping). The non-raw-R2 metrics split by architecture: LSTM's log R2, MedAE, and Within20pct all move to a non-overlapping, clearly-better range (e.g. log R2 0.9000-0.9105 vs the original's 0.8738-0.8849) -- a real effect. TCN's log R2 (0.8365-range, essentially the SAME as the original's 0.8353-range -- the single-seed "worsening" reported earlier was itself just an unlucky draw, not a real degradation either way) and MedAE (heavy overlap) show no robust effect; only TCN's Within20pct (9.01-12.58 vs 6.45-8.09) clears the original's range. All of this -- real or not -- stays far below the control (raw R2 0.6858) and the adopted best (0.6994): the classical lead on duration is not explained by `skip_back` either way.

**PM100 memory -- the single-seed raw-R2 "worsening" for LSTM was seed noise; TCN's improvement is real.** LSTM's 3-seed raw R2 range (0.8559-0.9335) almost fully overlaps the original's (0.8709-0.9361) -- the seed-0 "worsening" (0.8932) reported earlier is not distinguishable from normal seed variance, confirming the caveat already added above. TCN's raw R2 range (0.8682-0.9263) sits essentially entirely above the original's (0.7869-0.8702, touching only at the boundary) -- a real improvement. Non-raw-R2 metrics: TCN's log R2 and Within20pct both move to clearly non-overlapping, better ranges; LSTM's move in the same direction but with overlapping ranges (not robust on their own). MedAE for both architectures has too much within-policy spread (original std 5.83-9.37) to call either way from 3 seeds.

**Revised tally, from the 3-seed ranges (not the single-seed table any more)**: of the 12 duration/memory comparisons that are not raw R2 (log R2, raw MedAE, raw Within20pct, for LSTM and TCN), only **6 of 12 show a non-overlapping, seed-robust improvement under `skip_back`**: all 3 for duration LSTM, plus duration TCN's Within20pct and memory TCN's log R2 and Within20pct. The other 6 (duration TCN's log R2 and MedAE, all 3 of memory LSTM's, and memory TCN's MedAE) show overlapping ranges between `skip_back` and the original -- a mean-level trend in most cases, but not something 3 seeds can call a real effect. **So `skip_back` does NOT "consistently" improve the non-raw-R2 metrics** -- it reliably helps on all 3 metrics for duration LSTM, partially on one metric each for duration TCN and memory TCN, and shows no robust effect on any of the three for memory LSTM.

**Bottom line**: the only effects that survive a real multi-seed check, across all three combinations, are PM100 power's raw-R2 and log-R2 gains (clearest for LSTM, likely for TCN) and PM100 memory TCN's raw-R2/log-R2/Within20pct gains. Everything on PM100 duration's raw R2, and PM100 memory LSTM's raw R2, that looked like a `skip_back` effect in the single-seed table was seed noise. None of this changes the headline conclusion against classical ML's own adopted-best models.

**Not done here, stated plainly**: this multi-seed check reused each pair's already-found best lr (no new search) and base width only -- it is not a from-scratch re-tune under `skip_back`, so a real per-pair lr search might shift these ranges further. F-DATA is not retrained in this notebook; see `06h_fdata_sequence_tie_variant.ipynb` for the F-DATA skip_back results (a single fixed-lr run per pair there, for the reasons documented in that notebook).

## Multi-seed check: is `skip_back`'s single-run effect real, or seed noise?

The summary above flagged that two of the single-seed `skip_back` numbers (PM100 power LSTM
raw R2 0.9113, PM100 memory LSTM raw R2 0.8932) fall inside the ORIGINAL (`exclude`) policy's
own base-width seed-to-seed range from `06f_dl_comparison.ipynb`'s bounded sensitivity check --
so a single seed landing there is not yet evidence `skip_back` itself moved anything. This
section settles that by repeating all 6 `skip_back` pairs (PM100 power/duration/memory x
LSTM/TCN) at 3 seeds each (0, 1, 2 -- the same seeds `06f`'s sensitivity check used, for direct
comparability), reusing each pair's own already-found best learning rate from its `_skipback`
Optuna study above (no new hyperparameter search) via a 1-point `optuna.samplers.GridSampler`,
which runs `run_pair`'s exact training/early-stopping code path for a single fixed-lr trial --
not a reimplementation. `dl_pipeline.N_TRIALS` is temporarily set to 1 for this section only
(restored immediately after) so that single-trial call does not try to run 10 trials against a
1-point grid. Base architecture width only (matching every run elsewhere in this notebook); no
width dimension is tested here.

Also included: a second, independent repeat of the determinism check above, this time on PM100
duration's TCN pair (the first repeat used power TCN), to widen the evidence that the
determinism settings hold beyond the one pair already checked.

In [12]:
from src import dl_pipeline as dl_pipeline_module

DET_CHECK_COMBO2 = "pm100_duration"
prepared_det2 = skipback_prepared[DET_CHECK_COMBO2]
det_ckpt_dir2 = CKPT_DIR_SKIPBACK / "determinism_check_duration"
det_study_db2 = STUDIES_DIR / f"{DET_CHECK_COMBO2}_TCN_detcheck2.db"


def run_tcn_once_duration(seed):
    set_determinism(seed)
    if det_study_db2.exists():
        det_study_db2.unlink()
    if det_ckpt_dir2.exists():
        shutil.rmtree(det_ckpt_dir2)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = run_pair(prepared_det2, "TCN", device=DEVICE, ckpt_dir=det_ckpt_dir2,
                           study_suffix="_detcheck2", sampler=optuna.samplers.TPESampler(seed=seed))
    det_warnings = [str(wi.message) for wi in caught
                    if "determinist" in str(wi.message).lower()]
    return result, det_warnings


t0 = time.perf_counter()
result_c, warnings_c = run_tcn_once_duration(SEED)
print(f"run C (pm100_duration TCN) done in {time.perf_counter() - t0:.1f}s")

t0 = time.perf_counter()
result_d, warnings_d = run_tcn_once_duration(SEED)
print(f"run D (pm100_duration TCN) done in {time.perf_counter() - t0:.1f}s")

[I 2026-11-02 12:35:46,301] A new study created in RDB with name: pm100_duration_TCN_detcheck2


[I 2026-11-02 12:36:43,479] Trial 0 finished with value: 2.6561562024060454 and parameters: {'lr': 0.00012520653814999459}. Best is trial 0 with value: 2.6561562024060454.


[I 2026-11-02 12:37:43,520] Trial 1 finished with value: 2.4914682802657584 and parameters: {'lr': 0.0002693883019285409}. Best is trial 1 with value: 2.4914682802657584.


[I 2026-11-02 12:38:43,779] Trial 2 finished with value: 2.6020659481245634 and parameters: {'lr': 0.00016051911333587616}. Best is trial 1 with value: 2.4914682802657584.


[I 2026-11-02 12:39:44,056] Trial 3 finished with value: 2.576498708602574 and parameters: {'lr': 0.00012296071107325708}. Best is trial 1 with value: 2.4914682802657584.


[I 2026-11-02 12:40:44,331] Trial 4 finished with value: 2.5821625800277066 and parameters: {'lr': 7.035737028722141e-05}. Best is trial 1 with value: 2.4914682802657584.


... (14 outputs left out)

[I 2026-11-02 12:53:02,956] Trial 7 finished with value: 2.522814618485266 and parameters: {'lr': 0.0006074996073425694}. Best is trial 1 with value: 2.4914682802657584.


[I 2026-11-02 12:53:36,157] Trial 8 finished with value: 2.5697734395971468 and parameters: {'lr': 0.0008459126528049375}. Best is trial 1 with value: 2.4914682802657584.


[I 2026-11-02 12:54:36,593] Trial 9 finished with value: 2.69868451082051 and parameters: {'lr': 5.8463261216434157e-05}. Best is trial 1 with value: 2.4914682802657584.


run D (pm100_duration TCN) done in 566.6s


In [13]:
preds_equal2 = np.array_equal(result_c["preds_log"], result_d["preds_log"])
r2_equal2 = (result_c["raw_metrics"]["R2"] == result_d["raw_metrics"]["R2"]
             and result_c["log_metrics"]["R2"] == result_d["log_metrics"]["R2"])

print(f"Run C: log_R2={result_c['log_metrics']['R2']:.10f} raw_R2={result_c['raw_metrics']['R2']:.10f} "
      f"best_lr={result_c['best_lr']:.6e}")
print(f"Run D: log_R2={result_d['log_metrics']['R2']:.10f} raw_R2={result_d['raw_metrics']['R2']:.10f} "
      f"best_lr={result_d['best_lr']:.6e}")
print(f"\nTest predictions bit-identical (np.array_equal): {preds_equal2}")
print(f"R2 (log and raw) identical to full float precision: {r2_equal2}")

all_det_warnings2 = sorted(set(warnings_c + warnings_d))
if all_det_warnings2:
    print(f"\n{len(all_det_warnings2)} distinct determinism warning(s):")
    for msg in all_det_warnings2:
        print(f"  - {msg}")
else:
    print("\nNo determinism warnings were raised.")

DETERMINISTIC2 = bool(preds_equal2 and r2_equal2)
print(f"\nCONCLUSION (pm100_duration TCN, second pair checked): the two runs are "
      f"{'BIT-IDENTICAL' if DETERMINISTIC2 else 'NOT bit-identical'}.")

Run C: log_R2=0.8048639762 raw_R2=0.4740046162 best_lr=2.693883e-04
Run D: log_R2=0.8048639762 raw_R2=0.4740046162 best_lr=2.693883e-04

Test predictions bit-identical (np.array_equal): True
R2 (log and raw) identical to full float precision: True

No determinism warnings were raised.

CONCLUSION (pm100_duration TCN, second pair checked): the two runs are BIT-IDENTICAL.


In [14]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED_CHECK_SEEDS = [0, 1, 2]

seed_check_best_lr = {}
for combo in PM100_COMBOS:
    for arch in SKIPBACK_ARCHS:
        study_name = f"{combo}_{arch}_skipback"
        storage = f"sqlite:///{STUDIES_DIR / study_name}.db"
        study = optuna.load_study(study_name=study_name, storage=storage)
        seed_check_best_lr[(combo, arch)] = study.best_trial.params["lr"]
print("Reused best lr per pair (from the _skipback studies above, no new search):")
for k, v in seed_check_best_lr.items():
    print(f"  {k}: lr={v:.4e}")

all_seed_check_runs = [(combo, arch, seed) for combo in PM100_COMBOS
                        for arch in SKIPBACK_ARCHS for seed in SEED_CHECK_SEEDS]
print(f"\nTotal planned runs: {len(all_seed_check_runs)}")

Reused best lr per pair (from the _skipback studies above, no new search):
  ('pm100_power', 'LSTM'): lr=8.4591e-04
  ('pm100_power', 'TCN'): lr=6.0750e-04
  ('pm100_duration', 'LSTM'): lr=6.0750e-04
  ('pm100_duration', 'TCN'): lr=2.6939e-04
  ('pm100_memory_c4', 'LSTM'): lr=8.4591e-04
  ('pm100_memory_c4', 'TCN'): lr=8.4591e-04

Total planned runs: 18


In [15]:
def run_one_seed_check(combo, arch, seed, lr):
    """Single fixed-lr run via run_pair itself (not a reimplementation): a 1-point
    GridSampler plus dl_pipeline_module.N_TRIALS temporarily set to 1 makes run_pair's own
    study.optimize() run exactly one trial at the given lr, through the exact same
    training/early-stopping/checkpoint code path as every other run in this notebook."""
    set_determinism(seed)
    ckpt_dir = CKPT_DIR_SKIPBACK / "seed_check" / f"seed{seed}"
    result = run_pair(skipback_prepared[combo], arch, device=DEVICE, ckpt_dir=ckpt_dir,
                       study_suffix=f"_skipback_seedcheck_seed{seed}",
                       sampler=optuna.samplers.GridSampler({"lr": [lr]}))
    return result


print(f"dl_pipeline_module.N_TRIALS before: {dl_pipeline_module.N_TRIALS}")
dl_pipeline_module.N_TRIALS = 1

first_combo_sc, first_arch_sc, first_seed_sc = all_seed_check_runs[0]
t0 = time.perf_counter()
first_result_sc = run_one_seed_check(first_combo_sc, first_arch_sc, first_seed_sc,
                                      seed_check_best_lr[(first_combo_sc, first_arch_sc)])
first_elapsed_sc = time.perf_counter() - t0
estimated_total_sc = first_elapsed_sc * len(all_seed_check_runs)
print(f"First run ({first_combo_sc}, {first_arch_sc}, seed={first_seed_sc}): {first_elapsed_sc:.1f}s")
print(f"Naive estimate for all {len(all_seed_check_runs)} runs: {estimated_total_sc/60:.1f} min")

dl_pipeline_module.N_TRIALS before: 10


First run (pm100_power, LSTM, seed=0): 63.9s
Naive estimate for all 18 runs: 19.2 min


In [16]:
seed_check_rows = [dict(combo=first_combo_sc, arch=first_arch_sc, seed=first_seed_sc,
                         lr=seed_check_best_lr[(first_combo_sc, first_arch_sc)],
                         elapsed_s=first_elapsed_sc,
                         log_R2=first_result_sc["log_metrics"]["R2"],
                         raw_R2=first_result_sc["raw_metrics"]["R2"],
                         raw_MedAE=first_result_sc["raw_metrics"]["MedAE"],
                         raw_Within20pct=first_result_sc["raw_metrics"]["Within20pct"])]

for combo, arch, seed in all_seed_check_runs[1:]:
    lr = seed_check_best_lr[(combo, arch)]
    t0 = time.perf_counter()
    result = run_one_seed_check(combo, arch, seed, lr)
    elapsed = time.perf_counter() - t0
    seed_check_rows.append(dict(combo=combo, arch=arch, seed=seed, lr=lr, elapsed_s=elapsed,
                                 log_R2=result["log_metrics"]["R2"], raw_R2=result["raw_metrics"]["R2"],
                                 raw_MedAE=result["raw_metrics"]["MedAE"],
                                 raw_Within20pct=result["raw_metrics"]["Within20pct"]))
    print(f"{combo} {arch} seed={seed}: log_R2={result['log_metrics']['R2']:.4f} "
          f"raw_R2={result['raw_metrics']['R2']:.4f} ({elapsed:.1f}s)")

dl_pipeline_module.N_TRIALS = 10
print(f"\ndl_pipeline_module.N_TRIALS restored to: {dl_pipeline_module.N_TRIALS}")

seed_check_df = pd.DataFrame(seed_check_rows)
print(f"\nTotal seed-check wall time: {seed_check_df['elapsed_s'].sum():.0f}s")

pm100_power LSTM seed=1: log_R2=0.9393 raw_R2=0.9214 (64.3s)


pm100_power LSTM seed=2: log_R2=0.9378 raw_R2=0.9165 (64.5s)


pm100_power TCN seed=0: log_R2=0.9132 raw_R2=0.7961 (62.7s)


pm100_power TCN seed=1: log_R2=0.9241 raw_R2=0.8539 (62.4s)


pm100_power TCN seed=2: log_R2=0.9004 raw_R2=0.8285 (56.6s)


pm100_duration LSTM seed=0: log_R2=0.9062 raw_R2=0.6285 (51.9s)


... (7 outputs left out)

pm100_memory_c4 LSTM seed=2: log_R2=0.9586 raw_R2=0.8559 (63.9s)


pm100_memory_c4 TCN seed=0: log_R2=0.9663 raw_R2=0.9263 (61.9s)


pm100_memory_c4 TCN seed=1: log_R2=0.9671 raw_R2=0.8999 (62.3s)


pm100_memory_c4 TCN seed=2: log_R2=0.9624 raw_R2=0.8682 (61.9s)

dl_pipeline_module.N_TRIALS restored to: 10

Total seed-check wall time: 1115s


In [17]:
skipback_seed_summary = (seed_check_df.groupby(["combo", "arch"])
                          [["log_R2", "raw_R2", "raw_MedAE", "raw_Within20pct"]]
                          .agg(["mean", "std", "min", "max"]))
print("skip_back, 3 seeds (0,1,2), base width, fixed lr (reused from the _skipback study):")
display(skipback_seed_summary.round(4))

# Original (`exclude`) base-width sensitivity numbers, transcribed from
# 06f_dl_comparison.ipynb's own already-executed `sens_summary` cell output (3 seeds: 0,1,2) --
# not recomputed here, since that notebook's in-memory sensitivity_df is not persisted to a
# results file. Base width, LSTM/TCN only (FNN rows omitted -- FNN is not part of skip_back).
original_base_summary_rows = [
    dict(combo="pm100_power", arch="LSTM", log_R2_mean=0.9258, log_R2_std=0.0044, log_R2_min=0.9212, log_R2_max=0.9301,
         raw_R2_mean=0.8791, raw_R2_std=0.0299, raw_R2_min=0.8565, raw_R2_max=0.9131,
         raw_MedAE_mean=140.9128, raw_MedAE_std=5.4077, raw_MedAE_min=136.5697, raw_MedAE_max=146.9697,
         raw_Within20pct_mean=61.0227, raw_Within20pct_std=1.8159, raw_Within20pct_min=59.1555, raw_Within20pct_max=62.7826),
    dict(combo="pm100_power", arch="TCN", log_R2_mean=0.8844, log_R2_std=0.0016, log_R2_min=0.8829, log_R2_max=0.8861,
         raw_R2_mean=0.7899, raw_R2_std=0.0211, raw_R2_min=0.7660, raw_R2_max=0.8059,
         raw_MedAE_mean=154.0193, raw_MedAE_std=12.0238, raw_MedAE_min=145.2574, raw_MedAE_max=167.7273,
         raw_Within20pct_mean=53.0426, raw_Within20pct_std=0.5733, raw_Within20pct_min=52.4208, raw_Within20pct_max=53.5504),
    dict(combo="pm100_duration", arch="LSTM", log_R2_mean=0.8787, log_R2_std=0.0057, log_R2_min=0.8738, log_R2_max=0.8849,
         raw_R2_mean=0.4658, raw_R2_std=0.0374, raw_R2_min=0.4391, raw_R2_max=0.5086,
         raw_MedAE_mean=5.6528, raw_MedAE_std=0.6847, raw_MedAE_min=4.9078, raw_MedAE_max=6.2544,
         raw_Within20pct_mean=8.8727, raw_Within20pct_std=0.8664, raw_Within20pct_min=7.9234, raw_Within20pct_max=9.6209),
    dict(combo="pm100_duration", arch="TCN", log_R2_mean=0.8353, log_R2_std=0.0062, log_R2_min=0.8294, log_R2_max=0.8418,
         raw_R2_mean=0.4472, raw_R2_std=0.0514, raw_R2_min=0.3886, raw_R2_max=0.4848,
         raw_MedAE_mean=12.3952, raw_MedAE_std=0.6498, raw_MedAE_min=11.6711, raw_MedAE_max=12.9278,
         raw_Within20pct_mean=7.4106, raw_Within20pct_std=0.8569, raw_Within20pct_min=6.4479, raw_Within20pct_max=8.0899),
    dict(combo="pm100_memory_c4", arch="LSTM", log_R2_mean=0.9577, log_R2_std=0.0081, log_R2_min=0.9484, log_R2_max=0.9637,
         raw_R2_mean=0.9122, raw_R2_std=0.0359, raw_R2_min=0.8709, raw_R2_max=0.9361,
         raw_MedAE_mean=8.6105, raw_MedAE_std=5.8329, raw_MedAE_min=3.4394, raw_MedAE_max=14.9334,
         raw_Within20pct_mean=89.7793, raw_Within20pct_std=3.2060, raw_Within20pct_min=87.3218, raw_Within20pct_max=93.4058),
    dict(combo="pm100_memory_c4", arch="TCN", log_R2_mean=0.9481, log_R2_std=0.0073, log_R2_min=0.9420, log_R2_max=0.9562,
         raw_R2_mean=0.8305, raw_R2_std=0.0418, raw_R2_min=0.7869, raw_R2_max=0.8702,
         raw_MedAE_mean=14.2500, raw_MedAE_std=9.3715, raw_MedAE_min=3.9706, raw_MedAE_max=22.3180,
         raw_Within20pct_mean=85.8737, raw_Within20pct_std=1.7166, raw_Within20pct_min=83.8926, raw_Within20pct_max=86.9188),
]
original_base_summary = pd.DataFrame(original_base_summary_rows).set_index(["combo", "arch"])
print("\noriginal (`exclude`), 3 seeds (0,1,2), base width -- from 06f_dl_comparison.ipynb's "
      "own already-executed sens_summary output:")
display(original_base_summary.round(4))

control_rows = []
for combo in PM100_COMBOS:
    c = load_control(combo, "LightGBM")
    control_rows.append(dict(combo=combo, log_R2=c["log_metrics"]["R2"], raw_R2=c["raw_metrics"]["R2"],
                              raw_MedAE=c["raw_metrics"]["MedAE"], raw_Within20pct=c["raw_metrics"]["Within20pct"]))
print("\nsame-features LightGBM control (single fit, no seed sweep):")
display(pd.DataFrame(control_rows).set_index("combo").round(4))

skip_back, 3 seeds (0,1,2), base width, fixed lr (reused from the _skipback study):


log_R2                          raw_R2                         raw_MedAE                              raw_Within20pct                   \
                        mean     std     min     max    mean     std     min     max      mean      std       min       max            mean     std      min   
combo           arch                                                                                                                                           
pm100_duration  LSTM  0.9056  0.0052  0.9000  0.9105  0.5511  0.0718  0.4867  0.6285    3.5467   0.1301    3.3971    3.6328         13.6249  2.2818  11.7067   
                TCN   0.8365  0.0196  0.8184  0.8574  0.4926  0.0551  0.4442  0.5526   10.8846   2.0265    8.6077   12.4903         11.0688  1.8472   9.0107   
pm100_memory_c4 LSTM  0.9632  0.0041  0.9586  0.9661  0.8960  0.0389  0.8559  0.9335    4.5042   2.5024    2.9317    7.3898         92.0581  0.3227  91.7309   
                TCN   0.9653  0.0025  0.9624  0.9671  0.8981  0.0291  0.8682  0.9263    7.3811   6.2378    3.4713   14.5748         95.2329  0.2982  94.8903   
pm100_power     LSTM  0.9379  0.0014  0.9365  0.9393  0.9185  0.0026  0.9165  0.9214  132.2983   2.6028  130.1890  135.2070         64.1931  0.4731  63.7513   
                TCN   0.9126  0.0118  0.9004  0.9241  0.8262  0.0290  0.7961  0.8539  155.5263  13.9321  144.0028  171.0097         57.1035  3.2494  53.4376   

                               
                          max  
combo           arch           
pm100_duration  LSTM  16.1483  
                TCN   12.5832  
pm100_memory_c4 LSTM  92.3761  
                TCN   95.4338  
pm100_power     LSTM  64.6923  
                TCN   59.6288


original (`exclude`), 3 seeds (0,1,2), base width -- from 06f_dl_comparison.ipynb's own already-executed sens_summary output:


log_R2_mean  log_R2_std  log_R2_min  log_R2_max  raw_R2_mean  raw_R2_std  raw_R2_min  raw_R2_max  raw_MedAE_mean  raw_MedAE_std  \
combo           arch                                                                                                                                    
pm100_power     LSTM       0.9258      0.0044      0.9212      0.9301       0.8791      0.0299      0.8565      0.9131        140.9128         5.4077   
                TCN        0.8844      0.0016      0.8829      0.8861       0.7899      0.0211      0.7660      0.8059        154.0193        12.0238   
pm100_duration  LSTM       0.8787      0.0057      0.8738      0.8849       0.4658      0.0374      0.4391      0.5086          5.6528         0.6847   
                TCN        0.8353      0.0062      0.8294      0.8418       0.4472      0.0514      0.3886      0.4848         12.3952         0.6498   
pm100_memory_c4 LSTM       0.9577      0.0081      0.9484      0.9637       0.9122      0.0359      0.8709      0.9361          8.6105         5.8329   
                TCN        0.9481      0.0073      0.9420      0.9562       0.8305      0.0418      0.7869      0.8702         14.2500         9.3715   

                      raw_MedAE_min  raw_MedAE_max  raw_Within20pct_mean  raw_Within20pct_std  raw_Within20pct_min  raw_Within20pct_max  
combo           arch                                                                                                                     
pm100_power     LSTM       136.5697       146.9697               61.0227               1.8159              59.1555              62.7826  
                TCN        145.2574       167.7273               53.0426               0.5733              52.4208              53.5504  
pm100_duration  LSTM         4.9078         6.2544                8.8727               0.8664               7.9234               9.6209  
                TCN         11.6711        12.9278                7.4106               0.8569               6.4479               8.0899  
pm100_memory_c4 LSTM         3.4394        14.9334               89.7793               3.2060              87.3218              93.4058  
                TCN          3.9706        22.3180               85.8737               1.7166              83.8926              86.9188


same-features LightGBM control (single fit, no seed sweep):


,log_R2,raw_R2,raw_MedAE,raw_Within20pct
combo,,,,
pm100_power,0.9451,0.9013,127.0169,67.5744
pm100_duration,0.8684,0.6858,9.9748,12.0747
pm100_memory_c4,0.9801,0.9676,1.6774,96.4894
